In [0]:
%pip install transformers accelerate sentence-transformers chromadb


In [0]:
dbutils.library.restartPython()

In [0]:
%pip install --upgrade opentelemetry-api opentelemetry-sdk

In [0]:
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import chromadb


In [0]:
VECTOR_DB_PATH = "/local_disk0/tmp/chroma_db_test/"

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
collection = client.get_or_create_collection("legal_knowledge")

print(f"Collection ready. Vectors available: {collection.count()}")


In [0]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

qa_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0
)


In [0]:
def retrieve_chunks(query, k=5):
    if collection.count() == 0:
        return [], []

    query_vector = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_vector,
        n_results=k
    )

    return results["documents"][0], results["metadatas"][0]


In [0]:
def refine_context(docs):
    seen = set()
    refined = []

    for doc in docs:
        snippet = doc.strip()
        if not snippet:
            continue

        if snippet not in seen:
            refined.append(snippet)
            seen.add(snippet)

    return refined[:3]


In [0]:
def build_prompt(query, context):
    context_text = "\n\n".join(context) if context else "No legal context retrieved."

    prompt = f"""
You are a legal assistant helping Indian citizens understand laws.

Use the legal context below to answer the question clearly.
Explain in simple language.
Mention legal section numbers if available.
Include penalties if applicable.
Provide practical guidance.
Do not repeat the question or prompt text.

Question:
{query}

Legal Context:
{context_text}

Answer:
"""
    return prompt


In [0]:
def generate_answer(query):
    docs, metadata = retrieve_chunks(query)
    context = refine_context(docs)

    if not context:
        return "No relevant legal context found in the vector database.", metadata

    prompt = build_prompt(query, context)
    response = qa_model(prompt)[0]["generated_text"].strip()

    return response, metadata


In [0]:
def format_output(answer, metadata):
    sections = sorted({m.get("section", "") for m in metadata if m.get("section")})

    formatted = f"""
LEGAL EXPLANATION:

{answer}

Relevant Sections:
{", ".join(sections) if sections else "Refer to applicable legal provisions"}

Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""

    return formatted


In [0]:
query = "What is the penalty for not wearing a helmet?"

answer, metadata = generate_answer(query)

print(format_output(answer, metadata))
